In [2]:
pip install opencv-python easyocr numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2
import easyocr
import numpy as np
import re
from collections import defaultdict

In [4]:
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [81]:
def extract_number(text):
    text = text.replace(",", "").replace("%", "").replace("m", "").strip()

    # OCR sometimes reads zero as letter O
    text = text.replace("O", "0").replace("o", "0")

    matches = re.findall(r"-?\d+\.?\d*", text)

    if not matches:
        return None

    return float(matches[0])

In [82]:
def box_geometry(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]

    return {
        "left": min(xs),
        "right": max(xs),
        "top": min(ys),
        "bottom": max(ys),
        "cx": sum(xs) / 4,
        "cy": sum(ys) / 4
    }

In [83]:
def preprocess_for_ocr(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    return gray

In [84]:
def ocr_left_strip_for_percent_axis(image):
    h, w = image.shape[:2]
    
    # 5% width only — excludes data labels that start at ~7-8%
    strip = image[:, :int(w * 0.05)]
    
    if strip.size == 0:
        return []

    gray = cv2.cvtColor(strip, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    detected = []

    # Try two strategies — raw tends to drop % cleanly, CLAHE helps faint labels
    strategies = {
        "raw":  gray,
        "clahe": cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(gray.copy())
    }

    seen_cy = set()

    for name, img in strategies.items():
        # No % in allowlist — forces EasyOCR to drop it rather than hallucinate digits
        results = reader.readtext(img, allowlist='0123456789. m')
        for box, text, conf in results:
            if conf < 0.10:
                continue
            number = extract_number(text)
            if number is None:
                continue
            geo = box_geometry(box)
            cy_key = round(geo["cy"] / 2)  # back to original space
            if cy_key in seen_cy:
                continue
            seen_cy.add(cy_key)
            detected.append({
                "value":      number,
                "text":       text,
                "confidence": conf,
                "left":       geo["left"]   / 2,
                "right":      geo["right"]  / 2,
                "top":        geo["top"]    / 2,
                "bottom":     geo["bottom"] / 2,
                "cx":         geo["cx"]     / 2,
                "cy":         geo["cy"]     / 2,
                "box":        box
            })

    return detected

In [85]:
def ocr_numeric_boxes(image):
    processed = preprocess_for_ocr(image)
    results = reader.readtext(processed)

    numeric_boxes = []
    for box, text, confidence in results:
        if confidence < 0.2:
            continue
        number = extract_number(text)
        if number is None:
            continue
        geo = box_geometry(box)
        numeric_boxes.append({
            "value":      number,
            "text":       text,
            "confidence": float(confidence),
            "left":       geo["left"],
            "right":      geo["right"],
            "top":        geo["top"],
            "bottom":     geo["bottom"],
            "cx":         geo["cx"],
            "cy":         geo["cy"],
        })

    # Second pass: dedicated strip for faint gray percentage axis labels
    # that the main pass misses due to low contrast
    strip_detections = ocr_left_strip_for_percent_axis(image)

    existing_cy = {round(n["cy"]) for n in numeric_boxes}
    for item in strip_detections:
        # Only add if the main pass didn't already find something at this y-position
        if not any(abs(item["cy"] - cy) < 10 for cy in existing_cy):
            numeric_boxes.append(item)
            existing_cy.add(round(item["cy"]))

    processed_h, processed_w = processed.shape[:2]
    return numeric_boxes, processed_w, processed_h

In [86]:
# def remove_same_y_groups(numbers, image_width, image_height):
#     """
#     Removes numeric labels that look like stacked-bar/data labels.

#     Stacked/data labels often:
#     - share similar y-position
#     - spread across x-position

#     Y-axis labels usually:
#     - share similar x-position
#     - spread across y-position
#     """

#     if len(numbers) < 3:
#         return numbers

#     y_groups = defaultdict(list)
#     y_bin_size = image_height * 0.06

#     for item in numbers:
#         y_bin = int(item["cy"] // y_bin_size)
#         y_groups[y_bin].append(item)

#     labels_to_remove = set()

#     for group in y_groups.values():
#         if len(group) < 3:
#             continue

#         x_positions = [g["cx"] for g in group]
#         y_positions = [g["cy"] for g in group]

#         x_spread = max(x_positions) - min(x_positions)
#         y_spread = max(y_positions) - min(y_positions)

#         # Same y-level, spread horizontally = likely bar/data labels
#         if y_spread < image_height * 0.06 and x_spread > image_width * 0.25:
#             for g in group:
#                 labels_to_remove.add(id(g))

#     return [item for item in numbers if id(item) not in labels_to_remove]


In [87]:
def is_monotonic(values):
    if len(values) < 2:
        return False

    increasing = all(values[i] <= values[i + 1] for i in range(len(values) - 1))
    decreasing = all(values[i] >= values[i + 1] for i in range(len(values) - 1))

    return increasing or decreasing

In [88]:
def is_evenly_spaced(values, tolerance_ratio=0.20):
    if len(values) < 3:
        return False

    values = sorted(values)
    diffs = [values[i + 1] - values[i] for i in range(len(values) - 1)]

    if any(d == 0 for d in diffs):
        return False

    avg_diff = sum(diffs) / len(diffs)

    return all(abs(d - avg_diff) <= avg_diff * tolerance_ratio for d in diffs)

In [89]:
def find_vertical_value_axis(numbers, image_width, image_height):
    if len(numbers) < 3:
        return []

    groups = defaultdict(list)
    bin_size = image_width * 0.04

    for item in numbers:
        x_bin = int(item["right"] // bin_size)
        groups[x_bin].append(item)

    best_group = []
    best_score = 0

    for group in groups.values():
        if len(group) < 3:
            continue

        group = sorted(group, key=lambda x: x["cy"])
        values = [g["value"] for g in group]

        # Hard gate: must be evenly spaced to even be considered
        # This immediately rejects data labels and random number clusters
        if not is_evenly_spaced(values):
            continue

        x_positions = [g["right"] for g in group]
        y_positions = [g["cy"] for g in group]

        mean_x = sum(x_positions) / len(x_positions)
        x_spread = max(x_positions) - min(x_positions)
        y_spread = max(y_positions) - min(y_positions)

        score = 0

        if mean_x < image_width * 0.25:
            score += 3

        if x_spread < image_width * 0.12:
            score += 3

        if y_spread > image_height * 0.25:
            score += 3

        if len(group) >= 3:
            score += 2

        if is_monotonic(values):
            score += 4

        if is_evenly_spaced(values):
            score += 4

        if score > best_score and score >= 12:
            best_score = score
            best_group = group

    return best_group

In [90]:
def find_horizontal_value_axis(numbers, image_width, image_height):
    if len(numbers) < 3:
        return []

    groups = defaultdict(list)
    bin_size = image_height * 0.08

    for item in numbers:
        y_bin = int(item["cy"] // bin_size)
        groups[y_bin].append(item)

    best_group = []
    best_score = 0

    for group in groups.values():
        if len(group) < 3:
            continue

        group = sorted(group, key=lambda x: x["cx"])
        values = [g["value"] for g in group]

        # Hard gate: must be evenly spaced
        if not is_evenly_spaced(values):
            continue

        x_positions = [g["cx"] for g in group]
        y_positions = [g["cy"] for g in group]

        mean_y = sum(y_positions) / len(y_positions)
        x_spread = max(x_positions) - min(x_positions)
        y_spread = max(y_positions) - min(y_positions)

        score = 0

        # 1. x-axis labels sit near the bottom of the chart
        if mean_y > image_height * 0.75:
            score += 3

        # 2. x-axis labels should be spread horizontally
        if x_spread > image_width * 0.25:
            score += 3

        # 3. x-axis labels should be vertically tight
        if y_spread < image_height * 0.08:
            score += 3

        # 4. At least 3 tick values
        if len(group) >= 3:
            score += 2

        # 5. Values increase left to right
        if is_monotonic(values):
            score += 4

        # 6. Even spacing (already passed hard gate, so this always adds 4)
        if is_evenly_spaced(values):
            score += 4

        if score > best_score and score >= 12:
            best_score = score
            best_group = group

    return best_group

In [91]:
def ocr_bottom_left_for_zero(image, image_w, image_h):
    h, w = image.shape[:2]
    
    # Widen slightly to give EasyOCR more character separation context
    crop = image[int(h * 0.70):, :int(w * 0.25)]
    
    if crop.size == 0:
        return False

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    
    strategies = []
    
    # Strategy 1: raw grayscale — let EasyOCR handle contrast itself
    strategies.append(resized)
    
    # Strategy 2: CLAHE only, no threshold
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    strategies.append(clahe.apply(resized.copy()))
    
    # Strategy 3: Otsu threshold (background-aware)
    is_dark = np.mean(resized) < 128
    tt = cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU if is_dark else cv2.THRESH_BINARY + cv2.THRESH_OTSU
    _, s3 = cv2.threshold(resized.copy(), 0, 255, tt)
    strategies.append(s3)

    for img in strategies:
        results = reader.readtext(img, allowlist='0123456789.-')  # restrict alphabet
        for _, text, conf in results:
            num = extract_number(text)
            if conf > 0.05 and num is not None and abs(num) < 1.0:
                return True
    return False

In [92]:
def infer_zero_on_axis(axis_values, tolerance=0.01):
    if len(axis_values) < 2:
        return False
    vals = sorted(axis_values)
    step = (vals[-1] - vals[0]) / (len(vals) - 1)
    if step <= 0:
        return False
    extrapolated = vals[0] - step
    return abs(extrapolated) <= abs(step) * tolerance

In [93]:
def hunt_zero_in_full_image(image, image_width, image_height):
    """
    Dedicated pass to find a standalone '0' axis label.
    Restricts OCR alphabet and searches only the left strip.
    """
    h, w = image.shape[:2]
    left_strip = image[:, :int(w * 0.25)]  # only left 25%
    
    gray = cv2.cvtColor(left_strip, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    results = reader.readtext(gray, allowlist='0123456789.m%')
    for _, text, conf in results:
        num = extract_number(text)
        if conf > 0.05 and num is not None and abs(num) < 1.0:
            return True
    return False

In [94]:
def check_starts_at_zero(image_path, orientation):
    image = cv2.imread(image_path)

    if image is None:
        raise ValueError("Image not found")

    numbers, w, h = ocr_numeric_boxes(image)

    if orientation == "vertical":
        axis_group = find_vertical_value_axis(numbers, w, h)
        axis_type = "y-axis"

    elif orientation == "horizontal":
        axis_group = find_horizontal_value_axis(numbers, w, h)
        axis_type = "x-axis"

    else:
        return {
            "status": "unknown",
            "starts_at_zero": None,
            "reason": "Invalid or unknown orientation"
        }

    if not axis_group:
        return {
            "status": "compliant",
            "starts_at_zero": True,
            "orientation": orientation,
            "axis_type": axis_type,
            "detected_axis_values": [],
            "reason": "No value-axis labels detected; treated as compliant by client rule"
        }

    axis_values = [item["value"] for item in axis_group]

    contains_zero = any(abs(v) < 1.0 for v in axis_values)
    zero_source = "axis_group" if contains_zero else None

    if not contains_zero and orientation == "vertical":
        contains_zero = ocr_bottom_left_for_zero(image, w, h)
        if contains_zero:
            zero_source = "rescue_crop"
            axis_values = axis_values + [0.0]  # now visible in output

    if not contains_zero and orientation == "vertical":
        contains_zero = hunt_zero_in_full_image(image, w, h)
        if contains_zero:
            zero_source = "rescue_hunt"
            axis_values = axis_values + [0.0]  # now visible in output

    reason_map = {
        "axis_group":   "Zero found directly in axis label group",
        "rescue_crop":  "Zero found via bottom-left crop rescue",
        "rescue_hunt":  "Zero found via left-strip digit scan rescue",
        None:           "Value-axis labels detected but zero was not found"
    }

    return {
        "status": "compliant" if contains_zero else "non_compliant",
        "starts_at_zero": contains_zero,
        "orientation": orientation,
        "axis_type": axis_type,
        "detected_axis_values": sorted(axis_values),  # sorted so 0 shows at front
        "zero_source": zero_source,
        "reason": reason_map[zero_source]
    }

In [95]:
result = check_starts_at_zero(image_path="/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/Compliant/15.png", orientation="vertical")
print(result)

{'status': 'compliant', 'starts_at_zero': True, 'orientation': 'vertical', 'axis_type': 'y-axis', 'detected_axis_values': [], 'reason': 'No value-axis labels detected; treated as compliant by client rule'}


# Debugging

In [45]:
def debug_ocr(image_path):
    image = cv2.imread(image_path)
    h_orig, w_orig = image.shape[:2]
    
    # Show what the preprocessed image looks like
    processed = preprocess_for_ocr(image)
    cv2.imwrite("debug_preprocessed.png", processed)
    
    # Show ALL raw EasyOCR detections, no filtering
    results = reader.readtext(processed)
    print(f"Image original size: {w_orig}x{h_orig}")
    print(f"Preprocessed size: {processed.shape[1]}x{processed.shape[0]}")
    print(f"\nALL detections (including low confidence):")
    for box, text, conf in results:
        geo = box_geometry(box)
        print(f"  text='{text}' conf={conf:.2f} right={geo['right']:.0f} cy={geo['cy']:.0f}")

debug_ocr("//Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/Compliant/15.png")

Image original size: 1060x520
Preprocessed size: 2120x1040

ALL detections (including low confidence):
  text='10090' conf=0.39 right=100 cy=79
  text='Other' conf=0.42 right=2018 cy=89
  text='4296' conf=0.85 right=334 cy=96
  text='5.,696' conf=0.22 right=458 cy=101
  text='493' conf=0.39 right=582 cy=97
  text='6.626' conf=0.35 right=706 cy=105
  text='5.83' conf=0.66 right=828 cy=101
  text='4836' conf=0.89 right=952 cy=98
  text='6.346' conf=0.44 right=1076 cy=104
  text='6.296' conf=0.43 right=1200 cy=104
  text='675' conf=0.68 right=1326 cy=107
  text='21S' conf=0.02 right=1450 cy=108
  text='93E' conf=0.04 right=1574 cy=97
  text='6.280' conf=0.38 right=1820 cy=104
  text='6,630' conf=0.30 right=1944 cy=106
  text='8.896' conf=0.94 right=210 cy=114
  text='8 896' conf=0.43 right=1696 cy=114
  text='90' conf=0.09 right=1572 cy=151
  text='22.39' conf=0.44 right=588 cy=205
  text='22.036' conf=0.33 right=960 cy=203
  text='718.94' conf=0.13 right=1082 cy=203
  text='719 246' conf

In [46]:
def debug_rescue(image_path):
    image = cv2.imread(image_path)
    h, w = image.shape[:2]
    crop = image[int(h * 0.75):, :int(w * 0.20)]
    cv2.imwrite("debug_crop.png", crop)
    
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    gray = clahe.apply(gray)
    is_dark_bg = np.mean(gray) < 128
    thresh_type = cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU if is_dark_bg else cv2.THRESH_BINARY + cv2.THRESH_OTSU
    _, gray_thresh = cv2.threshold(gray, 0, 255, thresh_type)
    cv2.imwrite("debug_crop_thresh.png", gray_thresh)
    
    results = reader.readtext(gray_thresh)
    print(f"Crop size: {crop.shape}, dark_bg={is_dark_bg}")
    print("Rescue OCR results:")
    for _, text, conf in results:
        print(f"  text='{text}' conf={conf:.2f} -> number={extract_number(text)}")

debug_rescue("/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/Compliant/15.png")

Crop size: (130, 212, 3), dark_bg=False
Rescue OCR results:
  text='Wa' conf=0.00 -> number=None
  text='046' conf=0.82 -> number=46.0
  text='X' conf=0.37 -> number=None
  text='K' conf=0.25 -> number=None


In [25]:
def debug_zero_source(image_path):
    image = cv2.imread(image_path)
    numbers, w, h = ocr_numeric_boxes(image)
    print("All detected numbers:")
    for n in numbers:
        print(f"  value={n['value']} text='{n['text']}' right={n['right']:.0f} cy={n['cy']:.0f} conf={n['confidence']:.2f}")

debug_zero_source("/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/Compliant/10.png")

All detected numbers:
  value=0.0 text='Umsatzcrlose' right=164 cy=20 conf=0.71
  value=1004.0 text='1004' right=673 cy=97 conf=0.44
  value=916.0 text='0916' right=343 cy=124 conf=0.40
  value=965.0 text='965' right=453 cy=121 conf=0.90
  value=772.0 text='0772' right=227 cy=132 conf=0.73
  value=0.0 text='Feo' right=124 cy=438 conf=0.63
  value=0.0 text='Okt' right=562 cy=438 conf=0.26
  value=0.0 text='Doz' right=675 cy=439 conf=0.53


In [80]:
def debug_percent_strip(image_path):
    image = cv2.imread(image_path)
    h, w = image.shape[:2]
    
    left_strip = image[:, :int(w * 0.10)]
    cv2.imwrite("debug_strip_original.png", left_strip)
    
    gray = cv2.cvtColor(left_strip, cv2.COLOR_BGR2GRAY)
    gray2x = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    cv2.imwrite("debug_strip_2x.png", gray2x)
    
    clahe = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray2x.copy())
    cv2.imwrite("debug_strip_clahe.png", gray_clahe)
    
    print(f"Original image size: {w}x{h}")
    print(f"Strip size (original space): {int(w*0.10)}x{h}")
    print(f"Strip mean brightness: {gray.mean():.1f}")
    print()
    
    # Test same strip across 4 different clipLimits
    for clip in [2.5, 3.5, 5.0, 8.0]:
        clahe_test = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
        processed = clahe_test.apply(gray2x.copy())
        results = reader.readtext(processed, allowlist='0123456789.%m')
        print(f"clipLimit={clip}:")
        for _, text, conf in results:
            print(f"  text='{text}' conf={conf:.2f} -> number={extract_number(text)}")
        if not results:
            print("  (nothing detected)")
        print()
    
    # Also test with no CLAHE at all
    results_raw = reader.readtext(gray2x, allowlist='0123456789.%m')
    print("Raw (no CLAHE):")
    for _, text, conf in results_raw:
        print(f"  text='{text}' conf={conf:.2f} -> number={extract_number(text)}")
    if not results_raw:
        print("  (nothing detected)")

debug_percent_strip("/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/Compliant/15.png")

Original image size: 1060x520
Strip size (original space): 106x520
Strip mean brightness: 219.9

clipLimit=2.5:
  text='410080' conf=0.16 -> number=410080.0
  text='8.896' conf=0.95 -> number=8.896
  text='8019' conf=0.20 -> number=8019.0
  text='32.196' conf=0.66 -> number=32.196
  text='6010' conf=0.20 -> number=6010.0
  text='8.296' conf=0.74 -> number=8.296
  text='13106' conf=0.51 -> number=13106.0
  text='4080' conf=0.41 -> number=4080.0
  text='2080' conf=0.34 -> number=2080.0
  text='37.990' conf=0.77 -> number=37.99
  text='086' conf=0.33 -> number=86.0
  text='3' conf=0.22 -> number=3.0
  text='2023' conf=1.00 -> number=2023.0
  text='4' conf=0.69 -> number=4.0

clipLimit=3.5:
  text='410080' conf=0.18 -> number=410080.0
  text='28.896' conf=0.33 -> number=28.896
  text='8019' conf=0.19 -> number=8019.0
  text='32.196' conf=0.67 -> number=32.196
  text='6010' conf=0.30 -> number=6010.0
  text='8.296' conf=0.74 -> number=8.296
  text='13106' conf=0.48 -> number=13106.0
  text=